In [2]:
import pandas as pd

df=pd.read_csv('aluminium.csv')
print(df.shape)
df.head()

(1942, 375)


,date,target,LME_closed,cash_settlement_price_ma_5,cash_settlement_price_ma_10,cash_settlement_price_ma_20,cash_settlement_price_std_15,cash_settlement_price_rel_diff_weekly,cash_settlement_price_bollinger_bands_upper,cash_settlement_price_bollinger_bands_lower,...,_Lag_42_target,_Lag_49_target,_Lag_56_target,_Lag_63_target,_Lag_70_target,_Lag_77_target,_Lag_84_target,_Lag_90_target,_Lag_91_target,id
0,2018-01-01,2242.0,1,2213.4,2160.30,2088.125,87.854276,0.048889,2265.896845,1910.353155,...,2152.0,2144.0,2140.0,2139.5,2128.0,2065.0,2082.0,2097.0,2096.5,NaN
1,2018-02-06,2196.0,0,2213.7,2224.05,2215.475,19.213834,-0.014805,2267.310240,2163.639760,...,2165.0,2149.5,2144.0,2124.0,2116.0,2068.0,2069.5,2092.0,2097.0,NaN
2,2018-01-02,2256.0,0,2237.0,2179.80,2098.325,89.471637,0.055192,2290.218412,1906.431588,...,2152.0,2175.0,2111.0,2121.5,2117.0,2066.5,2075.0,2066.5,2092.0,NaN
3,2018-01-03,2241.0,0,2245.3,2194.70,2108.950,85.575322,0.018868,2307.964652,1909.935348,...,2131.0,2119.0,2106.5,2135.0,2102.5,2107.5,2072.0,2113.5,2066.5,NaN
4,2018-01-04,2230.0,0,2242.1,2206.85,2120.750,80.007187,-0.007124,2319.051870,1922.448130,...,2103.5,2143.0,2128.5,2133.0,2110.5,2164.0,2100.5,2113.5,2113.5,NaN


In [3]:
Cu=pd.read_csv("copper.csv")
Ni=pd.read_csv("nickel.csv")
print(Cu.shape)
print(Ni.shape)

(1942, 547)
(1942, 472)


# Checking null values
I need to replace the missing values if the script needs to work 

In [4]:
df.isnull().sum()

date                              0
target                           64
LME_closed                        0
cash_settlement_price_ma_5       64
cash_settlement_price_ma_10      64
                               ... 
_Lag_77_target                   64
_Lag_84_target                   64
_Lag_90_target                   64
_Lag_91_target                   64
id                             1878
Length: 375, dtype: int64

In [5]:
df = df.loc[:, df.isnull().mean() < 0.3]

In [6]:
df = df.fillna(method='ffill').fillna(method='bfill').fillna(df.mean(numeric_only=True))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5644\2989726357.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill').fillna(df.mean(numeric_only=True))


# Data Preparation

In [7]:
import numpy as np
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import NaNLabelEncoder, GroupNormalizer
from pytorch_forecasting.metrics import SMAPE
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["group_id"] = "aluminium"
df["time_idx"] = (df["date"] - df["date"].min()).dt.days
df = df.dropna(subset=["target"])

# Define max prediction horizon
max_prediction_length = 35  # 35 business days
max_encoder_length = 90     # lookback window

# Train-validation split
training_cutoff = df["time_idx"].max() - max_prediction_length

# Feature names
target = "target"
time_varying_known_reals = ["time_idx"] + [col for col in df.columns if "Lag" in col]
time_varying_unknown_reals = [target]

# Create TimeSeriesDataSet
training = TimeSeriesDataSet(
    df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target=target,
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_unknown_reals=time_varying_unknown_reals,
    time_varying_known_reals=time_varying_known_reals,
   target_normalizer=GroupNormalizer(groups=["group_id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    allow_missing_timesteps=True,   # 👈 ADD THIS LINE
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True) #validation set

batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=0)

ModuleNotFoundError: No module named 'pytorch_forecasting'

# Training the Temporal Fusion Transformer (TFT)

In [ ]:
from pytorch_lightning.strategies import DDPStrategy

early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")
lr_logger = LearningRateMonitor()

trainer = Trainer(
    max_epochs=30,
    accelerator="cpu",  # or "gpu" if CUDA is available
    devices=1,
    enable_checkpointing=True,
    callbacks=[early_stop, lr_logger],  # ✅ fixed name here
)

# Model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    loss=SMAPE(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)#

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [ ]:
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

TypeError: `model` must be a `LightningModule` or `torch._dynamo.OptimizedModule`, got `TemporalFusionTransformer`

In [ ]:
import pytorch_lightning as pl
import pytorch_forecasting as pf

print(pl.__version__)  # should be 1.9.5
print(pf.__version__)  # should be 0.10.3

2.5.1.post0
1.3.0


# Forecasting

In [ ]:
import matplotlib.pyplot as plt

# Get raw predictions and x safely
raw_output = tft.predict(validation, mode="raw", return_x=True)

# Inspect what raw_output[0] contains:
print(type(raw_output))          # Should be a list
print(type(raw_output[0]))       # Should be an object (dict or namedtuple-like)

# If it's a dict:
x = raw_output[0]['x']
raw_predictions = raw_output[0]['prediction']

# Plot
tft.plot_prediction(x, raw_predictions, idx=0)
plt.show()

<class 'pytorch_forecasting.models.base_model.Prediction'>
<class 'pytorch_forecasting.utils._utils.TupleOutputMixIn.to_network_output.<locals>.Output'>


AttributeError: 'Output' object has no attribute 'x'